# Phase 2 — Architecture 3: Legal Prefix–Conv LM

In [1]:
%pip -q install tokenizers cloudpickle
# Keep Kaggle's CUDA-enabled PyTorch.

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, socket, subprocess, sys, pickle, zipfile, shutil, gc
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import cloudpickle, torch
from torch import nn
from torch.nn import functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer

BASE = Path('/kaggle/input/datasets/abdakt/p2-a3')
DATA = Path('/kaggle/input/datasets/abdakt/p2-a3/Phase_2_LNER')
OUT = Path('/kaggle/working/p2_arch3'); OUT.mkdir(parents=True, exist_ok=True)
METHODS = ['full', 'partial', 'lora', 'lorma', 'adapter', 'kd']
C = json.loads((BASE/'config.json').read_text())
H = dict(seq=512, stride=384, micro=2, accum=4, seed=781, val_fraction=0.2,
         rank=16, alpha=16, adapter=64, last_layers=6, student_layers=9,
         epochs=dict(full=6, partial=8, lora=10, lorma=10, adapter=10, kd=8),
         lr=dict(full=2e-5, partial=4e-5, lora=3e-4, lorma=2e-4, adapter=3e-4, kd=3e-5),
         head_lr=3e-4, decay=0.01, patience=3, temperature=2., kd_weight=0.4)
print(' | '.join(torch.cuda.get_device_name(i) for i in range(2)))

Tesla T4 | Tesla T4


## Data and split

In [3]:
files = {p.name: p for p in DATA.rglob('*') if p.is_file()}
tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
tok.no_truncation(); tok.no_padding()
labels = files['labels.txt'].read_text().split()
tags = ['O'] + [f'{p}-{label}' for label in labels for p in ('B', 'I')]
tag_id = {t: i for i, t in enumerate(tags)}
pad = tok.token_to_id('<pad>')
texts, gold, train_ids, val_ids = {}, {}, [], []
rng = np.random.default_rng(H['seed'])
for fold in (1, 2):
    docs = json.loads(files[f'fold{fold}-text.json'].read_text())
    gold.update(json.loads(files[f'fold{fold}-gold.json'].read_text()))
    texts.update(docs)
    ids = sorted(docs); rng.shuffle(ids)
    n = round(len(ids)*H['val_fraction'])
    val_ids += ids[:n]; train_ids += ids[n:]
test_texts = json.loads(files['fold3-text.json'].read_text())
texts.update(test_texts)
records = {}
for uid, text in texts.items():
    enc = tok.encode(text, add_special_tokens=False)
    offsets = np.array(enc.offsets, dtype=np.int32).reshape(-1, 2)
    for j, (a, b) in enumerate(offsets):
        while a < b and text[a].isspace(): a += 1
        while b > a and text[b-1].isspace(): b -= 1
        offsets[j] = a, b
    y = np.zeros(len(enc.ids), dtype=np.int64)
    y[offsets[:, 0] == offsets[:, 1]] = -100
    for a, b, label, *rest in gold.get(uid, []):
        ix = np.flatnonzero((offsets[:, 1] > a) & (offsets[:, 0] < b) & (offsets[:, 1] > offsets[:, 0]))
        if len(ix):
            y[ix] = tag_id['I-'+label]; y[ix[0]] = tag_id['B-'+label]
    records[uid] = dict(ids=np.array(enc.ids, dtype=np.int64), offsets=offsets, y=y)
bundle = dict(records=records, gold=gold, texts=texts, train=train_ids, val=val_ids,
              test=list(test_texts), labels=labels, tags=tags, pad=pad)
with (OUT/'data.pkl').open('wb') as f: pickle.dump(bundle, f)
(OUT/'split.json').write_text(json.dumps(dict(train=train_ids, val=val_ids), indent=2))
(OUT/'settings.json').write_text(json.dumps(dict(model=C, tuning=H, tags=tags), indent=2))
shutil.copyfile(BASE/'tokenizer.json', OUT/'tokenizer.json')
print(f'{len(train_ids)} train | {len(val_ids)} validation | {len(test_texts)} test documents')

56 train | 14 validation | 35 test documents


## Architecture 3: Legal Prefix–Conv LM

In [4]:
# Architecture 3: Legal Prefix–Conv LM — RoPE, RMSNorm, SwiGLU, gated causal convolution
def norm(x, w):
    y = x.float()
    return (y * torch.rsqrt(y.square().mean(-1, keepdim=True) + 1e-6) * w).to(x.dtype)

def rope(x, cos, sin):
    a, b = x.float().chunk(2, -1)
    return torch.cat((a*cos-b*sin, b*cos+a*sin), -1).to(x.dtype)

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d, f = c['width'], c['ff']
        self.h = c['heads']
        self.p = nn.ParameterDict({k: nn.Parameter(torch.empty(*shape)) for k, shape in dict(
            n1=(d,), n2=(d,), nc=(d,), conv=(3,d), gate=(d,),
            qkv=(d,3*d), out=(d,d), up=(d,2*f), down=(f,d)).items()})
        for name, w in self.p.items():
            if name in ('n1','n2','nc'): nn.init.ones_(w)
            elif name == 'gate': nn.init.zeros_(w)
            else: nn.init.normal_(w, std=0.02 / math.sqrt(2*c['layers']) if name in ('out','down') else 0.02)

    def project(self, x, name):
        y = x @ self.p[name]
        if self.method in ('lora', 'lorma'):
            z = x if self.method == 'lora' else y
            y = y + (z @ self.low[name+'A'] @ self.low[name+'B']) * self.scale
        return y

    def forward(self, x, mask, cos, sin):
        p = self.p
        u = norm(x, p['nc'])
        history = x.new_zeros(x.size(0),2,x.size(-1))
        z = torch.cat((history,u),1)
        local = sum(z[:,2-i:2-i+x.size(1)] * p['conv'][i].to(x.dtype) for i in range(3))
        x = x + torch.sigmoid(u + p['gate'].to(x.dtype)) * local
        q, k, v = (a.view(x.size(0),x.size(1),self.h,-1).transpose(1,2)
                   for a in self.project(norm(x,p['n1']), 'qkv').chunk(3,-1))
        q, k = rope(q,cos,sin), rope(k,cos,sin)
        a = F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
        x = x + self.project(a.transpose(1,2).reshape_as(x), 'out')
        a, b = self.project(norm(x,p['n2']), 'up').chunk(2,-1)
        x = x + self.project(F.silu(a)*b, 'down')
        if self.method == 'adapter': x = x + self.adapter(x)
        return x

In [5]:
class NER(nn.Module):
    def __init__(self, c, h, method, nlabels):
        super().__init__()
        self.c, self.method = c, method
        self.embed = nn.Parameter(torch.empty(c['vocab'], c['width']))
        self.final = nn.Parameter(torch.ones(c['width']))
        self.blocks = nn.ModuleList([Block(c) for _ in range(c['layers'])])
        self.head = nn.Linear(c['width'], nlabels)
        self.drop = nn.Dropout(0.1)
        half = c['width']//c['heads']//2
        angle = torch.arange(h['seq']).float()[:, None] * 10000.**(-torch.arange(half).float()/half)
        self.register_buffer('cos', angle.cos()[None, None], persistent=False)
        self.register_buffer('sin', angle.sin()[None, None], persistent=False)
        for block in self.blocks:
            block.method = method
            if method in ('lora', 'lorma'):
                block.low = nn.ParameterDict(); block.scale = h['alpha']/h['rank']
                for name in ('qkv', 'out', 'up', 'down'):
                    din, dout = block.p[name].shape
                    n = din if method == 'lora' else dout
                    block.low[name+'A'] = nn.Parameter(torch.randn(n, h['rank']) * 0.02)
                    block.low[name+'B'] = nn.Parameter(torch.zeros(h['rank'], dout))
            if method == 'adapter':
                d = c['width']
                block.adapter = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, h['adapter']),
                                              nn.GELU(), nn.Linear(h['adapter'], d))
                nn.init.zeros_(block.adapter[-1].weight); nn.init.zeros_(block.adapter[-1].bias)

    def forward(self, ids, valid):
        x = F.embedding(ids, self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        mask = valid[:, None, None, :]
        for block in self.blocks:
            args = (x, mask, self.cos[:, :, :ids.size(1)], self.sin[:, :, :ids.size(1)])
            x = checkpoint(block, *args, use_reentrant=False) if self.training else block(*args)
        return self.head(self.drop(norm(x, self.final)))

def make_model(c, h, method, nlabels, base):
    cfg = dict(c)
    if method == 'kd': cfg['layers'] = h['student_layers']
    model = NER(cfg, h, method, nlabels)
    state = torch.load(Path(base)/'best.pt', map_location='cpu', weights_only=False)['model']
    if method == 'kd':
        layers = np.linspace(0, c['layers']-1, cfg['layers']).round().astype(int)
        state = {**{k: v for k, v in state.items() if not k.startswith('blocks.')},
                 **{f'blocks.{i}.'+k.split('.', 2)[2]: v
                    for i, j in enumerate(layers) for k, v in state.items() if k.startswith(f'blocks.{j}.')}}
    model.load_state_dict(state, strict=False)
    for name, p in model.named_parameters():
        p.requires_grad_(method in ('full', 'kd') or name.startswith('head.')
                         or (method == 'partial' and (name == 'final' or
                             (name.startswith('blocks.') and int(name.split('.')[1]) >= cfg['layers']-h['last_layers'])))
                         or '.low.' in name or '.adapter.' in name)
    return model

def load_tuned(path, c, h, nlabels, base):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    model = make_model(c, h, ck['method'], nlabels, base)
    model.load_state_dict(ck['model'], strict=False)
    return model

## Window batches and strict span F1

In [6]:
def windows(n, h):
    return list(range(0, max(n-h['seq'], 0)+1, h['stride'])) + ([n-h['seq']] if n > h['seq'] and (n-h['seq']) % h['stride'] else [])

class Rows(Dataset):
    def __init__(self, records, ids, h, pad):
        self.records, self.h, self.pad = records, h, pad
        self.rows = [(uid, start) for uid in ids for start in windows(len(records[uid]['ids']), h)]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        uid, start = self.rows[i]; row = self.records[uid]; s = self.h['seq']
        ids = row['ids'][start:start+s]; n = len(ids)
        x = torch.full((s,), self.pad, dtype=torch.long)
        y = torch.full((s,), -100, dtype=torch.long)
        x[:n] = torch.from_numpy(ids); y[:n] = torch.from_numpy(row['y'][start:start+s])
        return x, torch.arange(s) < n, y

def decode(scores, offsets, tags, text):
    spans, active = [], None
    for t, (a, b) in zip(scores.argmax(-1), offsets):
        a, b = int(a), int(b)
        if b <= a: continue
        tag = tags[int(t)]
        if tag == 'O' or tag[0] == 'B' or active is None or active[2] != tag[2:]:
            if active is not None: spans.append(active)
            active = None if tag == 'O' else [a, b, tag[2:]]
        else: active[1] = max(active[1], b)
    if active is not None: spans.append(active)
    return sorted({tuple(s) for s in spans})

@torch.inference_mode()
def predict(model, ids, data, h, device):
    model.eval(); result = {}
    for uid in ids:
        row = data['records'][uid]; n = len(row['ids'])
        if n == 0:
            result[uid] = []; continue
        scores = np.zeros((n, len(data['tags'])), dtype=np.float32)
        weights = np.zeros(n, dtype=np.float32)
        starts = windows(n, h)
        for i in range(0, len(starts), h['micro']):
            ss = starts[i:i+h['micro']]
            x = torch.full((len(ss), h['seq']), data['pad'], dtype=torch.long)
            valid = torch.zeros_like(x, dtype=torch.bool)
            for j, start in enumerate(ss):
                part = row['ids'][start:start+h['seq']]
                x[j, :len(part)] = torch.from_numpy(part); valid[j, :len(part)] = True
            with torch.autocast('cuda', dtype=torch.float16): logits = model(x.to(device), valid.to(device))
            logits = logits.float().cpu().numpy()
            for j, start in enumerate(ss):
                size = min(h['seq'], n-start)
                w = np.minimum(np.arange(size)+1, np.arange(size, 0, -1)).astype(np.float32)
                scores[start:start+size] += logits[j, :size]*w[:, None]
                weights[start:start+size] += w
        result[uid] = decode(scores/weights[:, None], row['offsets'], data['tags'], data['texts'][uid])
    return result

def strict_f1(pred, gold, labels):
    rows = []
    for label in labels:
        p = {(uid, int(a), int(b)) for uid, spans in pred.items() for a, b, lab in spans if lab == label}
        g = {(uid, int(e[0]), int(e[1])) for uid in pred for e in gold[uid] if e[2] == label}
        tp = len(p & g); precision = tp/max(len(p), 1); recall = tp/max(len(g), 1)
        rows.append(dict(label=label, precision=precision, recall=recall, f1=2*tp/max(len(p)+len(g), 1)))
    return float(np.mean([r['f1'] for r in rows])), rows

## DDP training

In [7]:
def train_worker(rank, world, c, h, base, out, method, refit=False):
    torch.set_num_threads(2); torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', rank=rank, world_size=world)
    torch.manual_seed(h['seed']); np.random.seed(h['seed'])
    out = Path(out)
    with (out/'data.pkl').open('rb') as f: data = pickle.load(f)
    run = out/method/('refit' if refit else 'validation'); run.mkdir(parents=True, exist_ok=True)
    train_ids = data['train'] + data['val'] if refit else data['train']
    epochs = json.loads((out/method/'validation/metrics.json').read_text())['best_epoch'] if refit else h['epochs'][method]
    model = make_model(c, h, method, len(data['tags']), base).to(device)
    total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, gradient_as_bucket_view=True)
    teacher = None
    if method == 'kd':
        teacher_path = out/'full'/('refit' if refit else 'validation')/'best.pt'
        teacher = load_tuned(teacher_path, c, h, len(data['tags']), base).to(device).half().eval()
        teacher.requires_grad_(False)
    dataset = Rows(data['records'], train_ids, h, data['pad'])
    sampler = DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True, seed=h['seed'])
    loader = DataLoader(dataset, batch_size=h['micro'], sampler=sampler, num_workers=0, pin_memory=True)
    counts = np.bincount(np.concatenate([data['records'][u]['y'][data['records'][u]['y'] >= 0] for u in train_ids]), minlength=len(data['tags']))
    weight = np.clip(np.sqrt(counts[1:].sum()/np.maximum(counts, 1)/max(len(counts)-1, 1)), 0.5, 3.)
    weight[0] = 0.35; weight = torch.tensor(weight, device=device, dtype=torch.float32)
    groups = [dict(params=[p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('head.')], lr=h['lr'][method]),
              dict(params=list(model.head.parameters()), lr=h['head_lr'])]
    opt = torch.optim.AdamW(groups, weight_decay=h['decay'], fused=True)
    scaler = torch.amp.GradScaler('cuda', init_scale=1024.)
    steps = math.ceil(len(loader)/h['accum'])*epochs; warmup = max(1, round(steps*0.1))
    best, best_epoch, stale, step, history = -1., 0, 0, 0, []
    torch.cuda.reset_peak_memory_stats(); started = time.monotonic()
    if rank == 0: print(f'{method} {"refit" if refit else "validation"}: {total:,} parameters | {trainable:,} trainable', flush=True)
    for epoch in range(1, epochs+1):
        sampler.set_epoch(epoch); ddp.train(); opt.zero_grad(set_to_none=True)
        running = torch.zeros(2, device=device)
        for i, (x, valid, y) in enumerate(loader):
            x, valid, y = (v.to(device, non_blocking=True) for v in (x, valid, y))
            group_size = min(h['accum'], len(loader)-(i//h['accum'])*h['accum'])
            last = (i+1) % h['accum'] == 0 or i+1 == len(loader)
            with nullcontext() if last else ddp.no_sync():
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = ddp(x, valid)
                    ce = F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), weight=weight, ignore_index=-100)
                    loss = ce
                    if teacher is not None:
                        with torch.no_grad(): soft = teacher(x, valid).float()
                        t = h['temperature']; mask = y != -100
                        kl = F.kl_div(F.log_softmax(logits.float()/t, -1), F.softmax(soft/t, -1), reduction='none').sum(-1)
                        kd = kl[mask].mean()*t*t
                        loss = (1-h['kd_weight'])*ce + h['kd_weight']*kd
                scaler.scale(loss/group_size).backward()
            running[0] += loss.detach(); running[1] += 1
            if last:
                factor = min((step+1)/warmup, 1.)*(0.1+0.9*(1+math.cos(math.pi*max(0, step-warmup)/max(1, steps-warmup)))/2)
                for group, lr in zip(opt.param_groups, (h['lr'][method], h['head_lr'])): group['lr'] = lr*factor
                scaler.unscale_(opt); nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        dist.all_reduce(running)
        control = torch.zeros(1, device=device)
        if not refit:
            pred = predict(model, data['val'][rank::world], data, h, device)
            gathered = [None]*world
            dist.all_gather_object(gathered, pred)
        if rank == 0:
            val_pred = {u: spans for part in gathered for u, spans in part.items()} if not refit else {}
            score, per_label = strict_f1(val_pred, data['gold'], data['labels']) if not refit else (None, [])
            improved = refit or score > best
            stale = 0 if improved else stale+1
            if improved:
                best, best_epoch = score, epoch
                names = {n for n, p in model.named_parameters() if p.requires_grad}
                state = {n: p.detach().cpu().half() for n, p in model.state_dict().items() if n in names}
                torch.save(dict(model=state, method=method, config=c, tuning=h, tags=data['tags'],
                                epoch=epoch, macro_f1=score, base_checkpoint=str(Path(base)/'best.pt')), run/'best.pt')
                if not refit:
                    (run/'validation_predictions.json').write_text(json.dumps(val_pred))
                    (run/'per_label.json').write_text(json.dumps(per_label, indent=2))
            history.append(dict(epoch=epoch, loss=(running[0]/running[1]).item(), macro_f1=score))
            metrics = dict(method=method, best_epoch=best_epoch, macro_f1=best, parameters=total,
                           trainable=trainable, seconds=time.monotonic()-started,
                           peak_gpu_gb=torch.cuda.max_memory_allocated()/2**30, history=history)
            (run/'metrics.json').write_text(json.dumps(metrics, indent=2))
            message = f'{method} epoch {epoch}/{epochs} | loss {history[-1]["loss"]:.4f}'
            if not refit: message += f' | strict Macro-F1 {score:.4f}'
            print(message, flush=True)
            control[0] = not refit and stale >= h['patience']
        dist.broadcast(control, 0)
        if control.item(): break
    dist.barrier()
    del ddp, model, opt, teacher, groups, logits, loss, ce, x, valid, y
    if method == 'kd': del soft, kl, kd
    gc.collect(); torch.cuda.empty_cache()
    model = load_tuned(run/'best.pt', c, h, len(data['tags']), base).to(device)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); infer_start = time.monotonic()
    pred = predict(model, data['test'][rank::world], data, h, device)
    torch.cuda.synchronize()
    infer = torch.tensor([time.monotonic()-infer_start, torch.cuda.max_memory_allocated()/2**30], device=device)
    dist.all_reduce(infer, op=dist.ReduceOp.MAX)
    gathered = [None]*world; dist.all_gather_object(gathered, pred)
    if rank == 0:
        pred = {u: spans for part in gathered for u, spans in part.items()}
        metrics = json.loads((run/'metrics.json').read_text())
        metrics.update(inference_seconds=infer[0].item(), inference_peak_gpu_gb=infer[1].item())
        (run/'metrics.json').write_text(json.dumps(metrics, indent=2))
        target = out/method
        (target/'prediction.json').write_text(json.dumps(pred, ensure_ascii=False))
        with zipfile.ZipFile(out/f'submission_{method}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
            z.write(target/'prediction.json', 'prediction.json')
        print(f'Saved submission_{method}.zip', flush=True)
    dist.barrier(); dist.destroy_process_group()

## Train and refit all six methods

In [8]:
def launch(function, *args):
    payload = OUT/'worker.pkl'
    with payload.open('wb') as f: cloudpickle.dump((function,args),f)
    with socket.socket() as s:
        s.bind(('127.0.0.1',0)); port = s.getsockname()[1]
    command = "import cloudpickle,os; f=open(os.environ['WORKER_PAYLOAD'],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(int(os.environ['RANK']),2,*args)"
    jobs = []
    for rank in range(2):
        env = dict(os.environ,MASTER_ADDR='127.0.0.1',MASTER_PORT=str(port),RANK=str(rank),
                   WORKER_PAYLOAD=str(payload),OMP_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
        jobs.append(subprocess.Popen([sys.executable,'-u','-c',command],env=env))
    while any(p.poll() is None for p in jobs):
        if any(p.poll() not in (None,0) for p in jobs):
            for p in jobs:
                if p.poll() is None: p.terminate()
            raise RuntimeError('GPU worker failed; see its traceback above.')
        time.sleep(0.5)
    if any(p.returncode for p in jobs): raise RuntimeError('GPU worker failed; see its traceback above.')

for method in METHODS:
    launch(train_worker, C, H, str(BASE), str(OUT), method, False)

full validation: 264,921,113 parameters | 264,921,113 trainable
full epoch 1/6 | loss 1.2078 | strict Macro-F1 0.2054
full epoch 2/6 | loss 0.1480 | strict Macro-F1 0.2610
full epoch 3/6 | loss 0.0553 | strict Macro-F1 0.2697
full epoch 4/6 | loss 0.0250 | strict Macro-F1 0.2901
full epoch 5/6 | loss 0.0128 | strict Macro-F1 0.2806
full epoch 6/6 | loss 0.0087 | strict Macro-F1 0.2821


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_full.zip
partial validation: 264,921,113 parameters | 77,139,993 trainable
partial epoch 1/8 | loss 1.4653 | strict Macro-F1 0.1471
partial epoch 2/8 | loss 0.2344 | strict Macro-F1 0.2165
partial epoch 3/8 | loss 0.1401 | strict Macro-F1 0.2466
partial epoch 4/8 | loss 0.0977 | strict Macro-F1 0.2560
partial epoch 5/8 | loss 0.0714 | strict Macro-F1 0.2578
partial epoch 6/8 | loss 0.0524 | strict Macro-F1 0.2587
partial epoch 7/8 | loss 0.0423 | strict Macro-F1 0.2621
partial epoch 8/8 | loss 0.0374 | strict Macro-F1 0.2675


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_partial.zip
lora validation: 269,713,433 parameters | 4,817,945 trainable
lora epoch 1/10 | loss 1.9310 | strict Macro-F1 0.1053
lora epoch 2/10 | loss 0.2845 | strict Macro-F1 0.2120
lora epoch 3/10 | loss 0.1508 | strict Macro-F1 0.2549
lora epoch 4/10 | loss 0.0934 | strict Macro-F1 0.2642
lora epoch 5/10 | loss 0.0578 | strict Macro-F1 0.2708
lora epoch 6/10 | loss 0.0358 | strict Macro-F1 0.2584
lora epoch 7/10 | loss 0.0211 | strict Macro-F1 0.2430
lora epoch 8/10 | loss 0.0133 | strict Macro-F1 0.2573


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_adapter.zip
kd validation: 149,251,097 parameters | 149,251,097 trainable
kd epoch 1/8 | loss 2.1881 | strict Macro-F1 0.1603
kd epoch 2/8 | loss 0.2178 | strict Macro-F1 0.2362
kd epoch 3/8 | loss 0.0799 | strict Macro-F1 0.2360
kd epoch 4/8 | loss 0.0448 | strict Macro-F1 0.2447
kd epoch 5/8 | loss 0.0318 | strict Macro-F1 0.2615
kd epoch 6/8 | loss 0.0266 | strict Macro-F1 0.2541
kd epoch 7/8 | loss 0.0246 | strict Macro-F1 0.2529
kd epoch 8/8 | loss 0.0235 | strict Macro-F1 0.2566


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_kd.zip


In [9]:
for method in METHODS:
    launch(train_worker, C, H, str(BASE), str(OUT), method, True)

full refit: 264,921,113 parameters | 264,921,113 trainable
full epoch 1/4 | loss 0.9448
full epoch 2/4 | loss 0.1208
full epoch 3/4 | loss 0.0493
full epoch 4/4 | loss 0.0285


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_full.zip
partial refit: 264,921,113 parameters | 77,139,993 trainable
partial epoch 1/8 | loss 1.3274
partial epoch 2/8 | loss 0.2124
partial epoch 3/8 | loss 0.1257
partial epoch 4/8 | loss 0.0833
partial epoch 5/8 | loss 0.0576
partial epoch 6/8 | loss 0.0402
partial epoch 7/8 | loss 0.0310
partial epoch 8/8 | loss 0.0265


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_partial.zip
lora refit: 269,713,433 parameters | 4,817,945 trainable
lora epoch 1/5 | loss 1.3551
lora epoch 2/5 | loss 0.2104
lora epoch 3/5 | loss 0.1240
lora epoch 4/5 | loss 0.0839
lora epoch 5/5 | loss 0.0652


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_adapter.zip
kd refit: 149,251,097 parameters | 149,251,097 trainable
kd epoch 1/5 | loss 1.5985
kd epoch 2/5 | loss 0.1520
kd epoch 3/5 | loss 0.0638
kd epoch 4/5 | loss 0.0435
kd epoch 5/5 | loss 0.0358


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


Saved submission_kd.zip


## Results and submissions

In [10]:
summary = [json.loads((OUT/m/'validation/metrics.json').read_text()) for m in METHODS]
print(f'{"Method":10} {"Macro-F1":>9} {"Epoch":>6} {"Params/M":>10} {"Infer sec":>10} {"Infer GB":>9}')
for row in summary:
    print(f'{row["method"]:10} {row["macro_f1"]:9.4f} {row["best_epoch"]:6} {row["parameters"]/1e6:10.1f} {row["inference_seconds"]:10.2f} {row["inference_peak_gpu_gb"]:9.2f}')
(OUT/'comparison.json').write_text(json.dumps(summary, indent=2))
with zipfile.ZipFile(OUT/'six_submissions.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for method in METHODS: z.write(OUT/f'submission_{method}.zip', f'submission_{method}.zip')
    for name in ('comparison.json', 'settings.json', 'split.json'): z.write(OUT/name, name)

Method      Macro-F1  Epoch   Params/M  Infer sec  Infer GB
full          0.2901      4      264.9      10.10      1.06
partial       0.2675      8      264.9      10.16      1.06
lora          0.2708      5      269.7      12.06      1.09
lorma         0.2904      9      271.1      12.30      1.10
adapter       0.2585      6      267.3      10.55      1.07
kd            0.2615      5      149.3       5.16      0.63


In [11]:
from collections import defaultdict

methods = ['full', 'partial', 'lora', 'lorma', 'adapter', 'kd']

scores = {
    m: json.loads((OUT/m/'validation/metrics.json').read_text())['macro_f1']
    for m in methods
}
val_preds = {
    m: json.loads((OUT/m/'validation/validation_predictions.json').read_text())
    for m in methods
}
ranked = sorted(methods, key=scores.get, reverse=True)

def ensemble(preds, selected, threshold):
    weights = {m: max(scores[m], 0.001) for m in selected}
    total = sum(weights.values())
    result = {}

    for uid in preds[selected[0]]:
        votes = defaultdict(float)
        for m in selected:
            for span in {tuple(e[:3]) for e in preds[m][uid]}:
                votes[span] += weights[m]

        candidates = sorted(
            (s for s in votes if votes[s]/total >= threshold),
            key=lambda s: (-votes[s], s)
        )
        kept = []
        for a, b, label in candidates:
            if not any(a < y and x < b for x, y, _ in kept):
                kept.append((a, b, label))
        result[uid] = sorted(kept)

    return result

In [12]:
best_f1 = -1.

for k in (3, 4, 6):
    selected = ranked[:k]
    for threshold in (0.35, 0.50, 0.65):
        pred = ensemble(val_preds, selected, threshold)
        f1, _ = strict_f1(pred, gold, labels)

        if f1 > best_f1:
            best_f1 = f1
            choice = dict(methods=selected, threshold=threshold)
            best_pred = pred

folder = OUT/'ensemble'
folder.mkdir(exist_ok=True)

(folder/'settings.json').write_text(json.dumps(
    dict(**choice, weights=scores, validation_macro_f1=best_f1), indent=2
))
(folder/'validation_predictions.json').write_text(json.dumps(best_pred))

print('Methods:', ', '.join(choice['methods']))
print(f'Threshold: {choice["threshold"]:.2f} | Validation Macro-F1: {best_f1:.4f}')
print(f'Best individual: {scores[ranked[0]]:.4f}')

Methods: lorma, full, lora, partial, kd, adapter
Threshold: 0.50 | Validation Macro-F1: 0.3379
Best individual: 0.2904


In [13]:
test_preds = {
    m: json.loads((OUT/m/'prediction.json').read_text())
    for m in choice['methods']
}
prediction = ensemble(test_preds, choice['methods'], choice['threshold'])

(folder/'prediction.json').write_text(
    json.dumps(prediction, ensure_ascii=False)
)
with zipfile.ZipFile(OUT/'submission_ensemble.zip', 'w',
                     zipfile.ZIP_DEFLATED) as z:
    z.write(folder/'prediction.json', 'prediction.json')

print('Saved:', OUT/'submission_ensemble.zip')

Saved: /kaggle/working/p2_arch3/submission_ensemble.zip


## Verification